In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql

CREATE OR REPLACE TABLE ${catalog}.silver.orders
USING DELTA
AS
 WITH orders_inc AS (
    SELECT *
    FROM ${catalog}.bronze.orders
),
orders_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM orders_inc
    ) t
    WHERE rn = 1
)
SELECT *
FROM orders_dedup;

In [0]:
%sql

CREATE OR REPLACE TABLE ${catalog}.silver.customers
USING DELTA
AS
 WITH customers_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.customers
    ) t
    WHERE rn = 1
)
SELECT *
FROM customers_dedup;

In [0]:
%sql

CREATE OR REPLACE TABLE ${catalog}.silver.products
USING DELTA
PARTITIONED BY (Category)
AS
 WITH products_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.products
    ) t
    WHERE rn = 1
)
SELECT *
FROM products_dedup;

In [0]:
%sql

CREATE OR REPLACE TABLE ${catalog}.silver.order_items
USING DELTA
AS
 WITH order_items_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_item_id, product_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.order_items
    ) t
    WHERE rn = 1
)
SELECT *
FROM order_items_dedup;

In [0]:
%sql
MERGE INTO ${catalog}.silver.orders AS target
USING (

WITH orders_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.orders
        WHERE CAST(created_at AS DATE)> (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE)) 
        OR CAST(updated_at AS DATE) > (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE))
        
    ) t
    WHERE rn = 1
)

SELECT * FROM orders_dedup

) AS source

ON target.order_id = source.order_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO ${catalog}.silver.customers AS target
USING (

WITH customers_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.customers
        WHERE CAST(created_at AS DATE)> (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE)) 
        OR CAST(updated_at AS DATE) > (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE))
    ) t
    WHERE rn = 1
)

SELECT * FROM customers_dedup

) AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO ${catalog}.silver.products AS target
USING (

WITH products_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.products
        WHERE CAST(created_at AS DATE)> (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE)) 
        OR CAST(updated_at AS DATE) > (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE))
    ) t
    WHERE rn = 1
)

SELECT * FROM products_dedup

) AS source

ON target.product_id = source.product_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO ${catalog}.silver.order_items AS target
USING (

WITH order_items_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_item_id, product_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.order_items
        WHERE CAST(created_at AS DATE)> (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE)) 
        OR CAST(updated_at AS DATE) > (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE))
    ) t
    WHERE rn = 1
)

SELECT * FROM order_items_dedup

) AS source

ON target.order_item_id = source.order_item_id
    AND target.product_id = source.product_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.silver.customer_orders
USING DELTA
AS
WITH orders_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.orders
    ) t
    WHERE rn = 1
),
customers_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.customers
    ) t
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.name,
    c.city,
    c.state,
    c.signup_date,
    o.order_id,
    o.order_date,
    o.total_amount,
    o.order_status
FROM orders_dedup o
LEFT JOIN customers_dedup c
    ON o.customer_id = c.customer_id



In [0]:
%sql
MERGE INTO ${catalog}.silver.customer_orders AS target
USING (

WITH orders_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM  ${catalog}.bronze.orders
        WHERE CAST(created_at AS DATE)> (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE)) 
        OR CAST(updated_at AS DATE) > (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE))
    ) t
    WHERE rn = 1
),
customers_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.customers
        WHERE CAST(created_at AS DATE)> (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE)) 
        OR CAST(updated_at AS DATE) > (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE))
    ) t
    WHERE rn = 1
    
)
SELECT
    c.customer_id,
    c.name,
    c.city,
    c.state,
    c.signup_date,
    o.order_id,
    o.order_date,
    o.total_amount,
    o.order_status
FROM orders_dedup o
LEFT JOIN customers_dedup c
    ON o.customer_id = c.customer_id
) AS source

ON target.order_id = source.order_id
AND target.customer_id = source.customer_id

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *
;

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalog}.silver.order_products
USING DELTA
AS
WITH products_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.products
    ) t
    WHERE rn = 1
),
order_items_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_item_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.order_items
    
    ) t
    WHERE rn = 1
)
SELECT /*+ BROADCAST(p) */
    p.product_id,
    p.product_name,
    p.category,
    o.order_id,
    o.order_item_id,
    o.quantity,
    o.price
FROM order_items_dedup o
LEFT JOIN products_dedup p
    ON o.product_id = p.product_id



In [0]:
%sql
MERGE INTO ${catalog}.silver.order_products AS target
USING (

WITH products_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.products
        WHERE CAST(created_at AS DATE)> (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE))
        OR CAST(updated_at AS DATE) > (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE)) 
    ) t
    WHERE rn = 1
),
order_items_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY order_item_id ORDER BY ingestion_timestamp DESC) AS rn
        FROM ${catalog}.bronze.order_items
        WHERE CAST(created_at AS DATE)> (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE)) 
        OR CAST(updated_at AS DATE) > (CAST (DATEADD(DAY,-1,current_timestamp()) AS DATE))
    
    ) t
    WHERE rn = 1
)
SELECT /*+ BROADCAST(p) */
    p.product_id,
    p.product_name,
    p.category,
    o.order_id,
    o.order_item_id,
    o.quantity,
    o.price
FROM order_items_dedup o
LEFT JOIN products_dedup p
    ON o.product_id = p.product_id
) AS source

ON target.order_id = source.order_id
AND target.product_id = source.product_id

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *
;

OPTIMEZE Z ORDER


In [0]:
%sql
OPTIMIZE ${catalog}.silver.orders ZORDER BY (order_id);

OPTIMIZE ${catalog}.silver.customers ZORDER BY (customer_id);

OPTIMIZE ${catalog}.silver.products ZORDER BY (product_id);

OPTIMIZE ${catalog}.silver.order_items ZORDER BY (order_id, product_id);

OPTIMIZE ${catalog}.silver.customer_orders ZORDER BY (order_id, customer_id);

OPTIMIZE ${catalog}.silver.order_products ZORDER BY (order_id, product_id);

In [0]:
%sql
VACUUM ${catalog}.silver.orders RETAIN 168 HOURS; 
VACUUM ${catalog}.silver.customers RETAIN 168 HOURS; 
VACUUM ${catalog}.silver.products RETAIN 168 HOURS;  
VACUUM ${catalog}.silver.order_items RETAIN 168 HOURS; 
VACUUM ${catalog}.silver.customer_orders RETAIN 168 HOURS; 
VACUUM ${catalog}.silver.order_products RETAIN 168 HOURS; 

Time Travel Validation

In [0]:
%sql
DESCRIBE HISTORY ${catalog}.silver.orders;

In [0]:
%sql
SELECT *
FROM ${catalog}.silver.orders VERSION AS OF 1;

Schema Enforcement

In [0]:
%sql
SET spark.databricks.delta.schema.autoMerge.enabled = true;